# MLPerf Inference — Whisper-large-v3 / LibriSpeech (Speech) on Colab

Runs **whisper-large-v3** on **LibriSpeech dev-clean** (the MLPerf speech2text model +
dataset + WER metric) on a Colab GPU, via `openai-whisper` (the master reference SUT uses
vLLM; openai-whisper gives the same model/data/metric more simply).

Expected WER ≈ **3–4%** on a dev-clean subset.

> Set **Runtime → Change runtime type → GPU** first.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('cuda',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2 · Dependencies (ffmpeg is preinstalled on Colab)

In [ ]:
!pip -q install "openai-whisper==20250625" "jiwer==4.0.0" "soundfile==0.14.0"
import whisper, jiwer; print('whisper + jiwer ready')

## 3 · LibriSpeech dev-clean (free, OpenSLR)

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -d LibriSpeech/dev-clean ]; then
  wget -q -O d.tgz https://www.openslr.org/resources/12/dev-clean.tar.gz
  # verify the archive BEFORE extracting
  echo "76f87d090650617fca0cac8f88b9416e0ebf80350acb97b343a85fa903728ab3  d.tgz" | sha256sum -c - \
    || { echo "!! dev-clean.tar.gz checksum mismatch"; rm -f d.tgz; exit 1; }
  tar xzf d.tgz && rm d.tgz
fi
echo 'utterances:' $(find LibriSpeech/dev-clean -name '*.flac' | wc -l)
# whisper.load_model('large-v3') verifies its own built-in SHA-256, so that fetch is already safe.

## 4 · WER eval (whisper-large-v3, N=30 utterances)

The Blackwell math-SDP guard is harmless on Colab's T4/L4 and keeps the notebook portable.

In [ ]:
%%writefile /content/whisper_eval.py
import os, glob, time, torch
try:
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
except Exception as e: print('SDP guard skipped:', e)
import whisper, jiwer
from whisper.normalizers import EnglishTextNormalizer
N=int(os.environ.get('N_UTT','30')); ROOT='/content/LibriSpeech/dev-clean'
refs={}
for t in glob.glob(os.path.join(ROOT,'*','*','*.trans.txt')):
    for line in open(t):
        u,x=line.strip().split(' ',1); refs[u]=x
items=[]
for f in sorted(glob.glob(os.path.join(ROOT,'*','*','*.flac'))):
    u=os.path.splitext(os.path.basename(f))[0]
    if u in refs: items.append((f,refs[u]))
items=items[:N]; print('utterances:',len(items))
model=whisper.load_model('large-v3', device='cuda' if torch.cuda.is_available() else 'cpu')
norm=EnglishTextNormalizer(); hyps=[]; gts=[]; asec=0.0; t0=time.time()
for f,ref in items:
    a=whisper.load_audio(f); asec+=len(a)/16000.0
    r=model.transcribe(a, language='en', fp16=torch.cuda.is_available(), verbose=False)
    hyps.append(norm(r['text'])); gts.append(norm(ref))
el=time.time()-t0; wer=jiwer.wer(gts,hyps)
print('================================================')
print(f'WER: {wer*100:.2f}%  ({len(items)} utts, LibriSpeech dev-clean)')
print(f'audio {asec:.1f}s | wall {el:.1f}s | RTF {el/asec:.3f}')
print('================================================')

In [ ]:
!cd /content && N_UTT=30 python whisper_eval.py

## Done ✅ — whisper-large-v3 WER on Colab GPU.